# Laboratori 12: Metropolis–Hastings dhe modeli Ising

**Lënda:** Simulim dhe Metoda Numerike  
**Kohëzgjatja:** 120 minuta  
**Mjedisi kryesor:** Python/NumPy; pjesa e dytë përmban zbatimin thelbësor në GNU Octave.

**Pyetja udhëheqëse:** Si kampionohet një shpërndarje që njihet vetëm deri në një konstante dhe si kontrollohet ekuilibri?

Ky laborator nuk duhet lexuar si një listë komandash. Çdo pjesë nis nga modeli dhe nga një llogaritje që mund të kryhet me dorë; kodi përdoret më pas për të zgjeruar llogaritjen, për ta verifikuar dhe për të eksploruar regjime që nuk trajtohen dot me disa hapa algjebrikë. Ritmi i punës është

$$\text{pyetje}\longrightarrow\text{model}\longrightarrow\text{derivim}\longrightarrow\text{algoritëm}\longrightarrow\text{kod}\longrightarrow\text{kontroll}.$$


## Rezultatet e të nxënit

Në fund të seancës studenti duhet të jetë në gjendje të:

- nxjerrë probabilitetin e pranimit Metropolis–Hastings;
- kontrollojë balancën e detajuar;
- zbatojë algoritmin në hapësirë njëpërmasore;
- përdorë logaritmet për qëndrueshmëri numerike;
- llogarisë autokorrelacionin dhe madhësinë efektive;
- nxjerrë ndryshimin lokal të energjisë në modelin Ising;
- llogarisë koston e një sweep-i dhe të paraqesë dinamikën e rrjetës.


## Organizimi i seancës

| Koha | Veprimtaria | Produkti |
|---:|---|---|
| 0–15 min | Pyetja fizike dhe llogaritja e parë me dorë | Parashikim i shkruar |
| 15–40 min | Derivimi dhe algoritmi minimal | Formula dhe pseudokodi |
| 40–65 min | Zbatimi i udhëhequr në Python | Rezultat i verifikuar |
| 65–82 min | Kompleksiteti: formulë dhe matje | $T(n)$ dhe grafik kohor |
| 82–100 min | Qëndrueshmëria dhe diagnostika | Krahasim i dy rasteve |
| 100–112 min | Shembulli i plotë dhe punishte | Eksperiment i modifikuar |
| 112–120 min | GNU Octave dhe ekzekutimi lokal | Skript i riprodhueshëm |

Seksioni i shembullit të plotë është i detyrueshëm. Në punishte zgjidhen vetëm një ose dy ndryshime parametrash gjatë orës; pjesa tjetër mbetet për punë të pavarur.


## Si duhet lexuar ky laborator

Metropolis–Hastings nuk jep mostra të pavarura dhe nuk fillon domosdoshmërisht në ekuilibër. Prandaj zbatimi i formulës së pranimit është vetëm gjysma e punës; gjysma tjetër është diagnostika. Çdo rezultat duhet të shoqërohet nga gjurma, norma e pranimit, termalizimi, autokorrelacioni dhe madhësia efektive.

Në modelin Ising duhet të dallohen hapat elementarë, sweep-et dhe mostrat statistike. Një sweep përmban $L^2$ propozime, por konfigurimet pas sweep-eve të njëpasnjëshme mbeten të korreluara, veçanërisht pranë pikës kritike.


## 1. Pse nevojitet MCMC

Synojmë një dendësi

$$\pi(x)=\frac{\widetilde\pi(x)}{Z},$$

ku $Z$ mund të jetë i panjohur. Metropolis–Hastings ndërton një zinxhir me shpërndarje stacionare $\pi$ pa llogaritur $Z$. Nga gjendja $x$ propozohet $x'\sim q(x'|x)$ dhe pranohet me

$$\alpha(x,x')=\min\left[1,
\frac{\widetilde\pi(x')q(x|x')}{\widetilde\pi(x)q(x'|x)}\right].$$

Për propozim simetrik, $q(x'|x)=q(x|x')$, raporti i propozimit anulohet.


## 2. Balanca e detajuar

Për kalimin $K(x,x')=q(x'|x)\alpha(x,x')$, zgjedhja e mësipërme siguron

$$\pi(x)K(x,x')=\pi(x')K(x',x).$$

Nëse raporti $r\leq1$, ana e majtë përmban faktor $r$ dhe bëhet e barabartë me anën e djathtë, ku kalimi i kundërt pranohet me probabilitet $1$; rasti $r>1$ është simetrik. Kjo është balanca e detajuar dhe garanton stacionaritetin nën kushte të përshtatshme të pakthyeshmërisë dhe aperodicitetit.


# Pjesa I — Python

## Përgatitja e mjedisit

Importet vendosen në një qelizë të vetme që eksperimenti të jetë i lexueshëm dhe i riprodhueshëm. Fara e gjeneratorit deklarohet edhe kur pjesa kryesore është deterministe; kështu çdo shtesë stokastike mund të përsëritet.


In [ ]:
import math
from time import perf_counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)  # realizim i riprodhueshëm
np.set_printoptions(precision=6, suppress=True)


## 3. Shembull njëpërmasor: normalja standarde

Përdorim $\log\widetilde\pi(x)=-x^2/2$ dhe propozimin $x'=x+\sigma_qZ$. Në vend të $\exp(\log\pi'-\log\pi)$, krahasojmë

$$\log U<\min(0,\log\widetilde\pi(x')-\log\widetilde\pi(x)).$$

Kjo shmang nënrrjedhjen kur dendësitë janë shumë të vogla.


In [ ]:
def metropolis_log(logpi,x0,N,sigma_q=1.0,rng=rng):
    x=np.empty(N); x[0]=x0; pranuar=0
    for n in range(1,N):
        propozim=x[n-1]+sigma_q*rng.normal()
        log_alpha=min(0.0,logpi(propozim)-logpi(x[n-1]))
        if np.log(rng.random())<log_alpha:
            x[n]=propozim; pranuar+=1
        else:
            x[n]=x[n-1]
    return x,pranuar/(N-1)

logpi=lambda x:-0.5*x*x
x,pa=metropolis_log(logpi,0.0,50_000,sigma_q=1.0)
print('pranimi=',pa,'mesatarja=',x[5000:].mean(),'devijimi=',x[5000:].std(ddof=1))


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(x[:1500],lw=.8); ax[0].set(xlabel='hapi',ylabel='x',title='Gjurmë')
ax[1].hist(x[5000:],bins=60,density=True,alpha=.7)
xx=np.linspace(-4,4,400); ax[1].plot(xx,np.exp(-xx**2/2)/np.sqrt(2*np.pi),'k')
for a in ax:a.grid(alpha=.3)
plt.show()


## Si do ta llogarisim kompleksitetin

Kompleksiteti nuk përcaktohet fillimisht me kronometër. Së pari zgjedhim madhësinë e hyrjes $n$, identifikojmë veprimin dominues dhe nxjerrim formulën $T(n)$. Vetëm pasi kemi një parashikim matematikor kryejmë kohëmatjen.

Për një cikël që kryen $c$ veprime në secilin prej $n$ hapave,

$$T(n)=cn+d\in\Theta(n).$$

Për dy cikle të plota të ndërfutura,

$$T(n)=\sum_{i=1}^{n}\sum_{j=1}^{n}c=cn^2\in\Theta(n^2).$$

Koha e matur modelohet si $t(n)\approx Cn^p$. Në koordinata log–log,

$$\log t=\log C+p\log n,$$

prandaj pjerrësia $p$ krahasohet me rendin e parashikuar. Matja mund të ndikohet nga kostot fikse, interpretuesi dhe memoria; ajo është kontroll empirik, jo zëvendësim i derivimit.


### 4.1 Kostoja dhe informacioni efektiv

Nëse vlerësimi i $\log\widetilde\pi$ kushton $C_\pi$, një zinxhir me $N$ hapa kushton

$$T(N)=\Theta(NC_\pi).$$

Por mostrat janë të korreluara. Nëse koha e integruar e autokorrelacionit është $\tau_{int}$,

$$N_{eff}\approx\frac{N}{\tau_{int}}.$$

Kostoja për mostër afërsisht të pavarur është, pra, proporcionale me $\tau_{int}C_\pi$. Një normë e lartë pranimi nuk është vetvetiu optimale: hapa shumë të vegjël pranohen shpesh, por eksplorojnë ngadalë.


In [ ]:
def acf_fft(x,max_lag=1000):
    x=np.asarray(x,float); x=x-x.mean(); n=len(x)
    m=1<<(2*n-1).bit_length(); f=np.fft.rfft(x,m)
    ac=np.fft.irfft(f*np.conjugate(f),m)[:n]
    ac/=np.arange(n,0,-1); ac/=ac[0]
    return ac[:max_lag+1]

for sq in [.1,.5,1.,3.,10.]:
    chain,pa=metropolis_log(logpi,0.,30_000,sq)
    rho=acf_fft(chain[3000:],500)
    K=np.argmax(rho<0) if np.any(rho<0) else len(rho)
    tau=1+2*np.sum(rho[1:K]); Neff=(len(chain)-3000)/tau
    print(f'sigma_q={sq:4.1f}: pranimi={pa:.3f}, tau={tau:.1f}, Neff={Neff:.0f}')


## Si do ta analizojmë qëndrueshmërinë

**Kushtëzimi** i përket problemit: sa ndryshon zgjidhja e saktë kur të dhënat perturbohen pak? **Qëndrueshmëria** i përket algoritmit: sa gabim shton procedura e llogaritjes? Në trajtë skematike,

$$\text{gabimi përpara}\lesssim
\text{numri i kushtëzimit}\times\text{gabimi prapa}.$$

Në secilin laborator do të përdorim një provë konkrete: mbetje, ligj ruajtjeje, kufi stabiliteti, rezultat analitik ose përsëritje me hap më të vogël. Fjalia “grafiku duket mirë” nuk është provë numerike.


### 5.1 Termalizimi, gjurma dhe zinxhirët e shumëfishtë

Pjesa fillestare varet nga $x_0$ dhe quhet termalizim. Heqja e një numri arbitrar pikash nuk provon ekuilibrin. Kontrollohen gjurma, mesataret në blloqe, autokorrelacioni dhe zinxhirë të nisur nga gjendje të ndryshme. Vlerësimet duhet të raportojnë $N$, pjesën e hequr, normën e pranimit dhe $N_{eff}$.

Qëndrueshmëria numerike kërkon përdorimin e log-dendësive; qëndrueshmëria statistike kërkon përzierje të mjaftueshme. Janë probleme të ndryshme.


In [ ]:
fillimet=[-8.,0.,8.]
fig,ax=plt.subplots(figsize=(7,4))
for x0 in fillimet:
    z,_=metropolis_log(logpi,x0,5000,1.0)
    ax.plot(np.cumsum(z)/np.arange(1,len(z)+1),label=f'x0={x0}')
ax.axhline(0,color='k',ls='--'); ax.set(xlabel='hapi',ylabel='mesatarja kumulative')
ax.grid(alpha=.3); ax.legend(); plt.show()


## 6. Modeli Ising dydimensional

Në një rrjetë katrore me spina $s_i=\pm1$ dhe kushte periodike,

$$E=-J\sum_{\langle i,j\rangle}s_is_j-H\sum_i s_i.$$

Kur kthehet vetëm spini $s_i$, ndryshojnë vetëm lidhjet me katër fqinjët:

$$\Delta E=2s_i\left(J\sum_{j\in fq(i)}s_j+H\right).$$

Në temperaturë $T$ dhe $k_B=1$, pranimi Metropolis është

$$\alpha=\min(1,e^{-\Delta E/T}).$$

Nëse $\Delta E\leq0$, lëvizja pranohet gjithmonë. Kjo formulë lokale shmang rillogaritjen e energjisë së gjithë rrjetës.


In [ ]:
def energji_ising(S,J=1.0,H=0.0):
    # Numërojmë vetëm fqinjët djathtas dhe poshtë që çdo lidhje të hyjë një herë.
    return -J*np.sum(S*(np.roll(S,-1,0)+np.roll(S,-1,1)))-H*np.sum(S)

def sweep_ising(S,T,J=1.0,H=0.0,rng=rng):
    L=S.shape[0]; pranuar=0
    for _ in range(L*L):
        i=rng.integers(L); j=rng.integers(L)
        fq=S[(i+1)%L,j]+S[(i-1)%L,j]+S[i,(j+1)%L]+S[i,(j-1)%L]
        dE=2*S[i,j]*(J*fq+H)
        if dE<=0 or np.log(rng.random()) < -dE/T:
            S[i,j]*=-1; pranuar+=1
    return pranuar/(L*L)

S=rng.choice([-1,1],size=(32,32))
print('E/site=',energji_ising(S)/S.size,'M/site=',S.mean())


## 7. Kompleksiteti i modelit Ising

Një sweep përmban $L^2$ propozime. Me formulën lokale të $\Delta E$, secili kushton $O(1)$:

$$T_{sweep}(L)=\Theta(L^2).$$

Për $S$ sweep-e, kostoja është $\Theta(SL^2)$ dhe kujtesa $\Theta(L^2)$. Rillogaritja e energjisë së plotë për çdo propozim do ta ngrinte koston në $\Theta(SL^4)$; ky është ndryshim algoritmik themelor.


In [ ]:
Ls=np.array([8,12,16,24,32,48]); ts=[]
for L in Ls:
    S=rng.choice([-1,1],size=(L,L))
    t0=perf_counter(); sweep_ising(S,2.5); ts.append(perf_counter()-t0)
print('pjerrësia ndaj L=',np.polyfit(np.log(Ls),np.log(ts),1)[0],'; pritja ≈2')


## 8. Shembulli i plotë dhe punishte dinamike

Krahasoni $T=1.5$, $T\approx2.27$ dhe $T=4$. Për çdo temperaturë përdorni termalizim, pastaj regjistroni energjinë dhe magnetizimin. Pranë temperaturës kritike shfaqen korrelacione të gjata dhe termalizimi ngadalësohet.

Kur `DINAMIK=True`, rrjeta përditësohet gjatë sweep-eve. Ndryshoni vetëm temperaturën ose madhësinë e rrjetës në një krahasim.


In [ ]:
def simulo_ising(L=32,T=2.3,term=300,matje=500,DINAMIK=False):
    S=rng.choice([-1,1],size=(L,L)); fig,ax=plt.subplots(figsize=(5,5))
    E=[]; M=[]; pr=[]
    for s in range(term+matje):
        pr.append(sweep_ising(S,T))
        if s>=term:
            E.append(energji_ising(S)/S.size); M.append(abs(S.mean()))
        if DINAMIK and s%10==0:
            ax.clear(); ax.imshow(S,cmap='coolwarm',vmin=-1,vmax=1); ax.set_title(f'T={T}, sweep={s}'); ax.axis('off'); plt.pause(.02)
    if not DINAMIK:
        ax.imshow(S,cmap='coolwarm',vmin=-1,vmax=1); ax.set_title(f'T={T}'); ax.axis('off')
    plt.show()
    return np.array(E),np.array(M),np.mean(pr)

for T in [1.5,2.3,4.0]:
    E,M,pa=simulo_ising(L=24,T=T,term=150,matje=250,DINAMIK=False)
    print(f'T={T}: <E>/N={E.mean():.3f}, <|M|>/N={M.mean():.3f}, pranim={pa:.3f}')


### Norma e pranimit dhe madhësia e hapit

Kur $\sigma_q$ është shumë i vogël, pothuajse çdo propozim pranohet, por zinxhiri lëviz ngadalë. Kur $\sigma_q$ është shumë i madh, propozimet bien shpesh në zona me dendësi të vogël dhe refuzohen. Qëllimi nuk është pranimi 100%, por një kompromis që rrit $N_{eff}$ për njësi kohe.


In [ ]:
sigmas=np.logspace(-1,1,12); acc=[]; ess=[]
for sq in sigmas:
    ch,pa=metropolis_log(logpi,0.,20000,sq); rho=acf_fft(ch[2000:],400)
    K=np.argmax(rho<0) if np.any(rho<0) else len(rho); tau=1+2*rho[1:K].sum()
    acc.append(pa); ess.append(18000/tau)
fig,ax=plt.subplots(1,2,figsize=(10,4)); ax[0].semilogx(sigmas,acc,'o-'); ax[1].semilogx(sigmas,ess,'o-')
ax[0].set(xlabel=r'$\sigma_q$',ylabel='pranimi'); ax[1].set(xlabel=r'$\sigma_q$',ylabel='$N_{eff}$')
for a in ax:a.grid(alpha=.3)
plt.show()


### Test njësi për ndryshimin lokal të energjisë

Formula $\Delta E$ është optimizimi qendror i kodit Ising. Para simulimit të gjatë duhet verifikuar duke zgjedhur disa vende, duke llogaritur energjinë e plotë para dhe pas kthimit dhe duke krahasuar diferencën me formulën lokale. Ky test zbulon gabime në kushtet periodike ose numërimin e fqinjëve.


In [ ]:
S_test=rng.choice([-1,1],size=(8,8))
for _ in range(10):
    i,j=rng.integers(8,size=2); fq=S_test[(i+1)%8,j]+S_test[(i-1)%8,j]+S_test[i,(j+1)%8]+S_test[i,(j-1)%8]
    dE_local=2*S_test[i,j]*fq; E0=energji_ising(S_test); S_test[i,j]*=-1; E1=energji_ising(S_test); S_test[i,j]*=-1
    assert np.isclose(E1-E0,dE_local)
print('10 kontrolle lokale të energjisë kaluan.')


### Madhësia e fundme dhe afërsia me pikën kritike

Në një rrjetë të fundme nuk ka singularitet të vërtetë; kalimi zbutet dhe varet nga $L$. Pranë $T_c$ domenet e mëdha dhe ngadalësimi kritik rrisin autokorrelacionin. Krahasimi i temperaturave duhet të përdorë termalizim dhe gjatësi matjeje të mjaftueshme, jo të njëjtin numër minimal sweep-esh pa diagnostikë.


In [ ]:
# Skanim i shkurtër pedagogjik; jo vlerësim preciz i T_c.
for T in np.linspace(1.5,4.0,6):
    E,M,pa=simulo_ising(L=16,T=float(T),term=100,matje=150,DINAMIK=False)
    print(f'T={T:.2f}, <|M|>={M.mean():.3f}, chi_proxy={16**2*M.var()/T:.3f}, pranimi={pa:.3f}')


### Pyetje për vetëkontroll

1. Pse pranimi shumë i lartë mund të jetë shenjë e një propozimi joefikas?
2. Çfarë problemi zgjidh përdorimi i logaritmeve?
3. Pse formula lokale e $\Delta E$ ndryshon kompleksitetin nga $L^4$ në $L^2$ për sweep?
4. Pse $N$ konfigurime nuk përfaqësojnë domosdoshmërisht $N$ mostra të pavarura?


# Pjesa II — GNU Octave

## 9. Metropolis për normalen

```octave
rand('seed',2026); randn('seed',2026);
N=50000; sigmaq=1; x=zeros(N,1); pr=0;
for n=2:N
    xp=x(n-1)+sigmaq*randn();
    logalpha=min(0,-0.5*xp^2+0.5*x(n-1)^2);
    if log(rand())<logalpha, x(n)=xp; pr=pr+1; else, x(n)=x(n-1); end
end
z=x(5001:end);
fprintf('pranimi %.4f, mesatarja %.4f, sigma %.4f\n',pr/(N-1),mean(z),std(z));
```

## 10. Bërthama Ising

```octave
L=24; T=2.3; S=2*(rand(L)>0.5)-1;
for sweep=1:500
    for k=1:L*L
        i=randi(L); j=randi(L);
        ip=mod(i,L)+1; im=mod(i-2,L)+1; jp=mod(j,L)+1; jm=mod(j-2,L)+1;
        fq=S(ip,j)+S(im,j)+S(i,jp)+S(i,jm);
        dE=2*S(i,j)*fq;
        if dE<=0 || log(rand())<-dE/T, S(i,j)=-S(i,j); end
    end
    if mod(sweep,10)==0, imagesc(S); axis equal tight off; drawnow; end
end
fprintf('|M|/N = %.5f\n',abs(mean(S(:))));
```


## Ekzekutimi jashtë notebook-ut

Një notebook e ndihmon shpjegimin, por studenti duhet të dijë të ekzekutojë edhe një program të zakonshëm. Qeliza vijuese krijon një skript minimal në `scripts_local/`. Ai përmban eksperimentin qendror dhe mund të ndryshohet me një editor teksti.


In [ ]:
script_dir = Path('scripts_local')
script_dir.mkdir(exist_ok=True)
script_text = "import numpy as np\nrng=np.random.default_rng(2026)\nx=np.zeros(50000); acc=0\nfor n in range(1,len(x)):\n p=x[n-1]+rng.normal(); la=min(0.,-.5*p*p+.5*x[n-1]**2)\n if np.log(rng.random())<la: x[n]=p; acc+=1\n else: x[n]=x[n-1]\nprint('accept=',acc/(len(x)-1),'mean=',x[5000:].mean(),'std=',x[5000:].std())\n"
script_path = script_dir / 'laboratori_12_metropolis_ising.py'
script_path.write_text(script_text, encoding='utf-8')
print('U krijua:', script_path)


Ekzekutimi në terminal:

```bash
python3 scripts_local/laboratori_12_metropolis_ising.py
```

Mos raportoni vetëm se programi u ekzekutua. Shënoni komandën, një dalje numerike, një kontroll dhe një përfundim.

## Raporti i shkurtër dhe kontrolli përfundimtar

Raporti duhet të përmbajë pyetjen, modelin, supozimet, derivimin kryesor, analizën e kompleksitetit, provën e qëndrueshmërisë, figurën me boshte e njësi dhe interpretimin fizik.

- [ ] Përdora log-raportin e pranimit.
- [ ] Raportova termalizimin, pranimin dhe $N_{eff}$.
- [ ] Nxora $\Delta E$ lokalisht.
- [ ] Llogarita koston $\Theta(SL^2)$ dhe jo vetëm numrin e sweep-eve.
- [ ] Ekzekutova skriptin Python nga terminali.
- [ ] Identifikova fragmentin përkatës në GNU Octave.
